# Лекція 7: Python Web Server Integrations — Async, HTTPX, Testing, Practical MCP

**Курс**: Прикладна розробка програмного забезпечення на Python 2026

---

## Передумови

Ви вже знаєте:
- **Лекція 1**: Типи даних, змінні, f-рядки, venv, pip
- **Лекція 2**: Модель пам'яті, мутабельність, колекції, цикли та умови
- **Лекція 3**: Глибоке занурення в колекції, comprehensions, функції, args/kwargs
- **Лекція 4**: Lambda, генератори, декоратори, модулі, винятки, type hints
- **Лекція 5**: ООП, `@dataclass`, `@property`, JSON/CSV серіалізація, `pathlib`
- **Лекція 6**: HTTP, REST, FastAPI, Pydantic-схеми, `uv`, `ruff`/`black`, MCP-концепції

> 💡 FastAPI-скелет проєкту `notes-api` з Лекції 6 — це база!)

**Необхідні інструменти**: Python 3.13+, `pipx`, MCP-сумісний LLM-клієнт (Claude Desktop / Cursor / Gemini CLI)

> 📁 **Де знаходиться проєкт `notes-api`?**
>
> Проєкт `notes-api` живе в одному місці для всіх лекцій: **`project/notes-api/`** (від кореня репозиторію курсу).
>
> Код, який ви бачите в цьому ноутбуці — це **демонстрація**. Під час або після лекції **скопіюйте та перепишіть** відповідні файли у свій власний репозиторій проєкту. Саме так працюють реальні розробники: дивляться на приклад → розуміють → пишуть свою версію.
>
> ```
> your-repo/
> └── project/
>     └── notes-api/       ← ваш робочий проєкт
>         ├── app/
>         ├── tests/
>         ├── Makefile
>         └── pyproject.toml
> ```

## Цілі навчання

Після цієї лекції ви зможете:

1. **Пояснити, як працює event loop** (цикл подій) та різницю між `def` і `async def` у FastAPI
2. **Виконувати HTTP-запити** з `httpx` — синхронно та асинхронно, з обробкою таймаутів і помилок
3. **Налаштувати конфігурацію** проєкту через `pydantic-settings` та `.env`-файли
4. **Встановити та налаштувати MCP-сервер** (`keep-mcp`) і підключити його до LLM-клієнта
5. **Писати тести** з `pytest` та `TestClient` для FastAPI-ендпоінтів, включаючи мокінг через `monkeypatch`

---

## 1. Основи асинхронності

Ваш FastAPI-скелет готовий. Але у ньому є `async def` — що це означає і навіщо?

### 1.1 Аналогія: офіціант у ресторані

Уявіть ресторан з одним офіціантом:

| Синхронний офіціант | Асинхронний офіціант |
|---------------------|---------------------|
| Приймає замовлення столика №1 | Приймає замовлення столика №1 |
| **Стоїть біля кухні і чекає** | Йде приймати замовлення столика №2 |
| Приносить страву столику №1 | Приймає замовлення столика №3 |
| Тільки тоді йде до столика №2 | Кухня готова — приносить страву №1 |
| **Знову чекає біля кухні...** | Кухня готова — приносить страву №2 |

Результат: **один** асинхронний офіціант обслуговує стільки ж клієнтів, скільки **три** синхронних.

> 💡 **Event loop** (цикл подій) — це "менеджер ресторану", який каже офіціанту: "кухня ще готує — йди обслугови інший столик".

![Event Loop](https://cdn.hackersandslackers.com/2021/04/async_eventloop.jpg)

### 1.2 `time.sleep()` vs `asyncio.sleep()` — блокуючий vs неблокуючий виклик

In [ ]:
import time

# Синхронний (blocking) підхід — офіціант СТОЇТЬ біля кухні
start = time.perf_counter()

time.sleep(1)  # Блокує потік на 1 секунду
time.sleep(1)  # Ще 1 секунда
time.sleep(1)  # Ще 1 секунда

elapsed = time.perf_counter() - start
print(f"Синхронно: {elapsed:.2f} сек (очікуємо ~3 сек)")

In [ ]:
import asyncio
import time


async def async_demo():
    """Асинхронний (non-blocking) підхід — офіціант обслуговує інших."""
    start = time.perf_counter()

    # Запускаємо три "очікування" ОДНОЧАСНО
    await asyncio.gather(
        asyncio.sleep(1),  # Столик №1 чекає
        asyncio.sleep(1),  # Столик №2 чекає
        asyncio.sleep(1),  # Столик №3 чекає
    )

    elapsed = time.perf_counter() - start
    print(f"Асинхронно: {elapsed:.2f} сек (очікуємо ~1 сек)")


await async_demo()

3 секунди проти 1 секунди — ось сила асинхронності. Три задачі **чекали паралельно**, а не послідовно.

### 1.3 `def` vs `async def` у FastAPI

FastAPI підтримує обидва варіанти:

```python
# Синхронний — FastAPI запустить у thread pool (окремий потік)
@app.get("/sync")
def sync_endpoint():
    return {"mode": "sync"}

# Асинхронний — FastAPI запустить в event loop
@app.get("/async")
async def async_endpoint():
    return {"mode": "async"}
```

| Тип функції | Де працює | Коли використовувати |
|-------------|-----------|---------------------|
| `def` | Thread pool | Блокуючі операції (файли, синхронні бібліотеки) |
| `async def` | Event loop | Асинхронні операції (`await httpx`, `await asyncio.sleep`) |

> ⚠️ **Критичне правило**: НІКОЛИ не використовуйте блокуючі виклики всередині `async def`!

```python
# ❌ НЕПРАВИЛЬНО — time.sleep() блокує event loop!
@app.get("/bad")
async def bad_endpoint():
    time.sleep(5)  # Весь сервер "завис" на 5 секунд!
    return {"status": "slow"}

# ✅ ПРАВИЛЬНО — asyncio.sleep() не блокує
@app.get("/good")
async def good_endpoint():
    await asyncio.sleep(5)  # Event loop вільний для інших запитів
    return {"status": "fast"}
```

In [ ]:
# Перетворення sync → async: покроковий приклад

import asyncio
import time


# Крок 1: Синхронна функція
def fetch_data_sync():
    time.sleep(0.5)  # Імітація мережевого запиту
    return {"data": "from sync"}


# Крок 2: Асинхронна версія
async def fetch_data_async():
    await asyncio.sleep(0.5)  # Неблокуюче очікування
    return {"data": "from async"}


# Порівняння: два послідовних виклики
start = time.perf_counter()
r1 = fetch_data_sync()
r2 = fetch_data_sync()
print(f"Sync: {time.perf_counter() - start:.2f}с → {r1}, {r2}")


async def run_async():
    start = time.perf_counter()
    r1, r2 = await asyncio.gather(fetch_data_async(), fetch_data_async())
    print(f"Async: {time.perf_counter() - start:.2f}с → {r1}, {r2}")


await run_async()

![Async meme](https://external-preview.redd.it/i-built-my-own-asyncio-to-understand-how-async-i-o-works-v0--E8ZJ1UcwvH-Ml3MB_mSnYkrPfijjbEynlsYfIR_KkU.jpg?width=1080&crop=smart&auto=webp&s=0c66e513cd53c869dd8474f4e554c37238e6e56b)

---

## 2. HTTP-клієнт з httpx

**httpx** — це сучасна бібліотека для HTTP-запитів з Python. Вона має **однаковий API** для синхронних та асинхронних запитів і є основою `TestClient` у FastAPI.

```bash
uv add httpx
```

### 2.1 Синхронний запит

In [ ]:
import httpx

# Синхронний GET-запит — як requests, але сучасніший
response = httpx.get(
    "https://jsonplaceholder.typicode.com/posts/1",
    timeout=5.0,  # Завжди вказуйте таймаут!
)

print(f"Status: {response.status_code}")
print(f"Content-Type: {response.headers['content-type']}")

data = response.json()
print(f"Title: {data['title'][:50]}...")
print(f"User ID: {data['userId']}")

### 2.2 Асинхронний запит з `AsyncClient`

Для `async def` ендпоінтів у FastAPI використовуйте `httpx.AsyncClient`:

In [ ]:
import httpx
import asyncio
import time


async def fetch_posts():
    """Отримати кілька постів ОДНОЧАСНО через AsyncClient."""
    async with httpx.AsyncClient(timeout=5.0) as client:
        # Три запити паралельно — як три столики в ресторані
        start = time.perf_counter()
        responses = await asyncio.gather(
            client.get("https://jsonplaceholder.typicode.com/posts/1"),
            client.get("https://jsonplaceholder.typicode.com/posts/2"),
            client.get("https://jsonplaceholder.typicode.com/posts/3"),
        )
        elapsed = time.perf_counter() - start

        for resp in responses:
            data = resp.json()
            print(f"  Post #{data['id']}: {data['title'][:40]}...")

        print(f"\n3 запити за {elapsed:.2f} сек (паралельно!)")


await fetch_posts()

### 2.3 Обробка помилок та таймаути

У реальному коді мережа **ненадійна** — завжди обробляйте помилки:

In [ ]:
import httpx

# 1. raise_for_status() — кидає виняток при 4xx/5xx
try:
    response = httpx.get(
        "https://jsonplaceholder.typicode.com/posts/99999",
        timeout=5.0,
    )
    response.raise_for_status()  # Кидає httpx.HTTPStatusError при помилці
    print(f"OK: {response.json()}")
except httpx.HTTPStatusError as e:
    print(f"HTTP помилка: {e.response.status_code}")

# 2. TimeoutException — сервер не відповів вчасно
try:
    response = httpx.get(
        "https://httpbin.org/delay/10",  # Затримка 10 секунд
        timeout=2.0,  # Але ми чекаємо лише 2
    )
except httpx.TimeoutException:
    print("Таймаут: сервер не відповів за 2 секунди")

# 3. ConnectError — сервер недоступний
try:
    response = httpx.get("http://localhost:59999", timeout=2.0)
except httpx.ConnectError:
    print("Помилка з'єднання: сервер недоступний")

---

## 3. Конфігурація проєкту

### 3.1 Проблема: захардкоджені значення

```python
# ❌ НЕПРАВИЛЬНО — значення "зашиті" в код
app = FastAPI(title="Notes API")
API_KEY = "sk-super-secret-key-12345"  # Секрет у коді!
DEBUG = True  # На продакшені теж True?
```

Проблеми:
- Секрети потрапляють у git-історію
- Різні значення для розробки, тестування, продакшену
- Щоб змінити порт — треба змінювати код

### 3.2 Простий спосіб: `python-dotenv` + `os.getenv`

```python
import os
from dotenv import load_dotenv

load_dotenv()  # Читає .env файл

APP_NAME = os.getenv("APP_NAME", "Notes API")  # default якщо немає
DEBUG = os.getenv("DEBUG", "false").lower() == "true"
PORT = int(os.getenv("PORT", "8000"))
```

Проблеми цього підходу:
- Ручне перетворення типів (`int()`, `.lower() == "true"`)
- Немає валідації при старті
- Немає автодоповнення в IDE

### 3.3 Кращий спосіб: `pydantic-settings`

```bash
uv add pydantic-settings
```

Ось **реальний файл** `app/config.py` з нашого проєкту `notes-api`:

In [ ]:
# app/config.py — реальний код з проєкту notes-api
from pydantic_settings import BaseSettings


class Settings(BaseSettings):
    app_name: str = "Notes API"
    debug: bool = False
    port: int = 8000

    model_config = {"env_file": ".env"}


settings = Settings()

# Демо: значення за замовчуванням
print(f"App: {settings.app_name}")
print(f"Debug: {settings.debug}")
print(f"Port: {settings.port}")
print(f"\nТип port: {type(settings.port).__name__} (автоматичне перетворення!)")

In [ ]:
# Як Settings використовується в app/main.py:
# from app.config import settings
#
# app = FastAPI(title=settings.app_name, version="0.1.0", debug=settings.debug)

# .env файл (НЕ комітити в git!):
env_content = """
APP_NAME=Notes API
DEBUG=true
PORT=3000
"""
print(".env файл:")
print(env_content)

# .env.example (комітити — показує які змінні потрібні):
env_example = """
APP_NAME=Notes API
DEBUG=false
PORT=8000
"""
print(".env.example файл (комітимо в git):")
print(env_example)

# .gitignore:
print(".gitignore (обов'язково):")
print(".env")

| Підхід | Типізація | Валідація | IDE-підтримка | Складність |
|--------|:---------:|:---------:|:-------------:|:----------:|
| `os.getenv` | Ручна | Немає | Немає | Мінімальна |
| `pydantic-settings` | Автоматична | При старті | Повна | Мінімальна |

> 💡 `pydantic-settings` додає **одну залежність** і дає автоматичну типізацію, валідацію при старті додатку та IDE-автодоповнення.

---

## 4. Практичний MCP — налаштування keep-mcp
У Лекції 6 ми вивчили **концепції** MCP: Host → Client → Server, три примітиви (Tools, Resources, Prompts). Тепер — практика.

### 4.1 Встановлення keep-mcp

[keep-mcp](https://github.com/feuerdev/keep-mcp) — MCP-сервер для Google Keep.

```bash
# Крок 1: Встановлення pipx (якщо немає)
pip install pipx
# Крок 1.1: Встановлення keep-mcp через pipx (ізольоване середовище)
pipx install keep-mcp
```

> 💡 `pipx` встановлює Python-інструменти в ізольовані середовища — не забруднює ваш глобальний Python.

### 4.2 Автентифікація: Google Master Token

keep-mcp використовує бібліотеку [gkeepapi](https://github.com/kiwiz/gkeepapi) для доступу до Google Keep.

**Крок 2**: Отримати master token для авторизації.

Детальна інструкція: [gkeepapi Authentication](https://github.com/kiwiz/gkeepapi?tab=readme-ov-file#authentication)

```bash
# Після отримання токена — зберегти в змінну середовища:
export GOOGLE_MASTER_TOKEN="your-token-here"
export GOOGLE_EMAIL="your-email@gmail.com"
```

> ⚠️ **Безпека**: Master token дає **повний доступ** до вашого Google-акаунту. Ніколи не комітьте його в git! Використовуйте `.env` або менеджер секретів.

### 4.3 Конфігурація LLM-клієнта

**Крок 3**: Налаштувати ваш LLM-клієнт (host) для підключення до keep-mcp (server).

**Claude Desktop** (основний варіант) — файл `claude_desktop_config.json`:

```json
{
  "mcpServers": {
    "keep-mcp": {
      "command": "pipx",
      "args": ["run", "keep-mcp"],
      "env": {
        "GOOGLE_MASTER_TOKEN": "your-token-here",
        "GOOGLE_EMAIL": "your-email@gmail.com"
      }
    }
  }
}
```

Розташування файлу:
- **macOS**: `~/Library/Application Support/Claude/claude_desktop_config.json`
- **Windows**: `%APPDATA%\Claude\claude_desktop_config.json`

**Cursor** (альтернатива):
```json
{
  "mcpServers": {
    "keep-mcp": {
      "command": "pipx",
      "args": ["run", "keep-mcp"]
    }
  }
}
```

**Gemini CLI** (альтернатива) — файл `~/.gemini/settings.json`:
```json
{
  "mcpServers": {
    "keep-mcp": {
      "command": "pipx",
      "args": ["run", "keep-mcp"]
    }
  }
}
```

### 4.4 Перевірка з'єднання

**Крок 4**: Після конфігурації перезапустіть LLM-клієнт і перевірте:

1. У Claude Desktop з'явиться іконка 🔌 з переліком MCP-серверів
2. Попросіть Claude: *"Покажи мої нотатки в Google Keep з міткою keep-mcp"*
3. Claude повинен запитати дозвіл на використання інструменту `find` або `get_note`

### 4.5 Усунення проблем (Troubleshooting)

| Проблема | Рішення |
|----------|--------|
| `pipx: command not found` | `pip install --user pipx` та додати до PATH |
| `keep-mcp not found` | `pipx install keep-mcp` (не `pip install`) |
| Token помилка | Перевірити `GOOGLE_MASTER_TOKEN` та `GOOGLE_EMAIL` |
| Claude не бачить MCP | Перезапустити Claude Desktop після зміни конфігу |
| `Connection refused` | Перевірити, що `pipx run keep-mcp` працює в терміналі |
| Нотатки не знаходяться | За замовчуванням keep-mcp бачить лише нотатки з міткою `keep-mcp` |

---

## 5. Тестування з pytest

### 5.1 Навіщо тестувати?

```
Без тестів:                          З тестами:
"Здається, працює" 🤞               "Точно працює" ✅
"Я перевірив в браузері"            "CI перевіряє при кожному коміті"
"Зміни нічого не зламали... мабуть" "Зміни пройшли 47 тестів"
```

### 5.2 Основи pytest

```bash
uv add --dev pytest
```

Правила pytest:
- Файли: `test_*.py` або `*_test.py`
- Функції: `def test_*()`
- Перевірки: `assert` (звичайний Python!)

```bash
# Запуск:
uv run pytest           # Всі тести
uv run pytest -v        # Детальний вивід
uv run pytest -x        # Зупинитися на першій помилці
```

### 5.3 FastAPI TestClient

`TestClient` дозволяє тестувати API **без запуску сервера**. Під капотом він використовує `httpx`.

Ось **реальний файл** `tests/test_health.py` з нашого проєкту:

In [ ]:
# tests/test_health.py — реальний код з проєкту notes-api

from fastapi.testclient import TestClient

from app.main import app

client = TestClient(app)


def test_health_returns_200():
    response = client.get("/health")
    assert response.status_code == 200
    data = response.json()
    assert "status" in data
    assert data["status"] == "ok"


# Запуск в Jupyter (зазвичай pytest робить це автоматично)
test_health_returns_200()
print("test_health_returns_200 PASSED")

Ось **реальний файл** `tests/test_notes.py` — тести для ендпоінтів нотаток:

In [ ]:
# tests/test_notes.py — реальний код з проєкту notes-api

import httpx
from fastapi.testclient import TestClient

from app.main import app

client = TestClient(app)


def test_create_note_returns_201():
    response = client.post(
        "/notes/create",
        json={"title": "Test Note", "content": "Hello world", "tags": ["demo"]},
    )
    assert response.status_code == 201
    data = response.json()
    assert data["title"] == "Test Note"
    assert data["content"] == "Hello world"
    assert "id" in data
    assert "created_at" in data


def test_create_note_invalid_returns_422():
    response = client.post("/notes/create", json={})
    assert response.status_code == 422


def test_search_notes_returns_200():
    response = client.post("/notes/search", json={"query": "test"})
    assert response.status_code == 200
    data = response.json()
    assert "results" in data
    assert "total" in data
    assert data["results"] == []
    assert data["total"] == 0


# Запуск в Jupyter
test_create_note_returns_201()
print("test_create_note_returns_201 PASSED")

test_create_note_invalid_returns_422()
print("test_create_note_invalid_returns_422 PASSED")

test_search_notes_returns_200()
print("test_search_notes_returns_200 PASSED")

### 5.4 Мокінг з `monkeypatch`

Коли тест залежить від зовнішнього сервісу (API, база даних), ми **мокаємо** (mock) виклик — підміняємо його фейковою реалізацією.

Ось **реальний тест** з `tests/test_notes.py` — мокаємо `httpx.get`:

In [ ]:
# Приклад monkeypatch — мокаємо httpx.get
# (з tests/test_notes.py нашого проєкту)

import httpx


def test_external_call_with_monkeypatch(monkeypatch):
    """Example: mock httpx.get so we don't hit the real network."""

    class FakeResponse:
        status_code = 200

        def json(self):
            return {"title": "Mocked post", "id": 1}

    monkeypatch.setattr(httpx, "get", lambda *args, **kwargs: FakeResponse())

    result = httpx.get("https://jsonplaceholder.typicode.com/posts/1")
    assert result.status_code == 200
    assert result.json()["title"] == "Mocked post"


# В Jupyter monkeypatch недоступний (це pytest fixture),
# але ми можемо показати принцип вручну:
original_get = httpx.get  # Зберігаємо оригінал


class FakeResponse:
    status_code = 200

    def json(self):
        return {"title": "Mocked post", "id": 1}


httpx.get = lambda *args, **kwargs: FakeResponse()  # Підміна!

result = httpx.get("https://does-not-matter.com/anything")
print(f"Status: {result.status_code}")
print(f"Data: {result.json()}")
print("Запит НЕ пішов у мережу — ми підмінили httpx.get!")

httpx.get = original_get  # Відновлюємо оригінал

### 5.5 Концепт: інтеграційні тести

| Тип тесту | Що перевіряє | Швидкість | Коли запускати |
|-----------|-------------|:---------:|---------------|
| **Unit** | Одна функція ізольовано | Швидко | Кожен коміт |
| **Integration** | Взаємодія компонентів (API + DB) | Повільно | CI/CD, перед релізом |

```python
# Позначаємо інтеграційний тест маркером:
import pytest

@pytest.mark.integration
def test_real_google_keep_connection():
    """Цей тест потребує реального токена."""
    ...

# Запуск тільки інтеграційних:
# pytest -m integration

# Запуск БЕЗ інтеграційних:
# pytest -m "not integration"
```

![Testing meme](https://i.programmerhumor.io/2022/09/programmerhumor-io-programming-memes-testing-memes-915e25d6a9e5164.jpg)

---

## 7. Makefile

### Makefile — єдина команда для всього

Замість запам'ятовувати три окремі команди, об'єднуємо їх в **Makefile**.

Ось **реальний Makefile** з нашого проєкту `notes-api`:

In [ ]:
# Makefile з проєкту notes-api
makefile_content = """
.PHONY: check fix test

check:
\truff check .
\tblack --check .
\tpytest

fix:
\truff check --fix .
\tblack .

test:
\tpytest
"""
print(makefile_content)

print("Використання:")
print("  make check  — перевірити лінтер + форматування + тести")
print("  make fix    — автоматично виправити стиль")
print("  make test   — запустити тільки тести")

---

## Підсумок

### Що ми вивчили сьогодні:

**1. Async** — Розділ 1:
- Event loop (цикл подій) — "менеджер ресторану", що розподіляє задачі
- `async def` + `await` — неблокуючі функції для I/O-операцій
- `asyncio.gather()` — паралельне виконання кількох задач
- Критичне правило: НІКОЛИ не використовувати `time.sleep()` всередині `async def`

**2. httpx** — Розділ 2:
- `httpx.get()` — синхронний запит (як `requests`)
- `httpx.AsyncClient` — асинхронні запити для `async def` ендпоінтів
- `raise_for_status()`, `TimeoutException` — обробка помилок

**3. Конфігурація** — Розділ 3:
- `pydantic-settings` + `BaseSettings` — типізована конфігурація з `.env`
- `.env` → `.gitignore`, `.env.example` → git

**4. MCP практика** — Розділ 4:
- `pipx install keep-mcp` → конфігурація LLM-клієнта → перевірка з'єднання
- Чотири кроки: встановити → авторизувати → налаштувати → протестувати

**5. Безпека** — Розділ 5:
- Safe mode за замовчуванням, `UNSAFE_MODE=true` — лише коли розумієте ризики
- Принцип найменших привілеїв (Principle of Least Privilege)
- Гігієна облікових даних: `.env` + `.gitignore`

**6. Тестування** — Розділ 6:
- `pytest`: `test_*.py`, `def test_*()`, `assert`
- `TestClient` — тестування FastAPI без запуску сервера
- `monkeypatch` — мокінг зовнішніх залежностей

**6. Makefile** — Розділ 7:
 - треба трішки побавитися, щоб встановити make на Windows) 

---

## Що далі?

### Лекція 8: Docker, PostgreSQL, Docker Compose

Ваш `notes-api` працює локально. Але як розгорнути його на сервері? Як додати базу даних?

- **Docker**: упаковка додатку в контейнер — "працює на моїй машині" → "працює скрізь"
- **PostgreSQL**: реальна база даних замість stub-відповідей
- **Docker Compose**: запуск API + бази даних однією командою

> 💡 Патерн `Settings` з `pydantic-settings` (Розділ 3) стане основою для конфігурації Docker: `DATABASE_URL`, `REDIS_URL` — все через `.env`!

---

## Джерела

### Async та Event Loop
- [Python asyncio — Official Docs](https://docs.python.org/3/library/asyncio.html) — Офіційна документація asyncio
- [Intro to Asynchronous Python with Asyncio](https://hackersandslackers.com/python-concurrency-asyncio/) - Блог про реалізацію асинхронності в Python 
- [FastAPI — Async](https://fastapi.tiangolo.com/async/) — Пояснення async/await у FastAPI

### httpx
- [httpx — Official Docs](https://www.python-httpx.org/) — Документація httpx
- [httpx — Async Support](https://www.python-httpx.org/async/) — Асинхронний клієнт

### Конфігурація
- [pydantic-settings — Official Docs](https://docs.pydantic.dev/latest/concepts/pydantic_settings/) — Документація pydantic-settings

### Тестування
- [pytest — Official Docs](https://docs.pytest.org/) — Документація pytest
- [FastAPI — Testing](https://fastapi.tiangolo.com/tutorial/testing/) — Тестування FastAPI

### MCP
- [MCP — Official Docs](https://modelcontextprotocol.io/) — Офіційна документація MCP
- [keep-mcp](https://github.com/feuerdev/keep-mcp) — MCP-сервер для Google Keep
- [gkeepapi](https://github.com/kiwiz/gkeepapi) — Python-бібліотека для Google Keep

### Інструменти
- [ruff — Official Docs](https://docs.astral.sh/ruff/) — Лінтер ruff
- [Black — Official Docs](https://black.readthedocs.io/en/stable/) — Форматер Black